# LLM Context Window Check

Check how many samples exceed the model's context window for each dataset.
Works with any model — just change `MODEL_PATH` below.

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [14]:
# ============================================================
# CONFIGURE HERE
# ============================================================
MODEL_PATH = "/raid_storage/shared_models/AceGPT-v2-8B"  # change to any model
# Set to None to read from model config automatically
CONTEXT_WINDOW_OVERRIDE = None  # e.g. 2048, 4096, 8192, or None

In [15]:
from transformers import AutoTokenizer, AutoConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
config = AutoConfig.from_pretrained(MODEL_PATH)

if CONTEXT_WINDOW_OVERRIDE is not None:
    CONTEXT_WINDOW = CONTEXT_WINDOW_OVERRIDE
else:
    # Try common config keys for context length
    CONTEXT_WINDOW = getattr(config, 'max_position_embeddings', None) \
        or getattr(config, 'max_sequence_length', None) \
        or getattr(config, 'seq_length', None) \
        or getattr(config, 'n_positions', None) \
        or 2048

MODEL_NAME = MODEL_PATH.rstrip('/').split('/')[-1]
print(f"Model: {MODEL_NAME}")
print(f"Context window: {CONTEXT_WINDOW}")
print(f"Vocab size: {tokenizer.vocab_size}")

TypeError: not a string

In [6]:
import json
import requests
import datasets
import numpy as np
from jinja2 import Environment, StrictUndefined
from tqdm.auto import tqdm

In [7]:
# Fetch all prompts from the API
prompts_data = None
for _ in range(10):
    resp = requests.get('https://promptlab.up.railway.app/api/prompt/list?project_secret_key=6Wirj')
    if resp.ok:
        prompts_data = resp.json()
        break
if not prompts_data:
    raise Exception('Failed to fetch prompts')

filtered_prompts = [
    p for p in prompts_data
    if p['status'] == 'APPROVED' and p['text_direction'].lower() == 'ltr'
]
print(f"Total approved LTR prompts: {len(filtered_prompts)}")

Total approved LTR prompts: 352


In [8]:
# Dataset configurations
# task_type: 'classification' splits on last line, 'generation' splits on |||
DATASET_CONFIGS = {
    # ---- Primary datasets (have tune.ipynb) ----
    'dialect_identification/AraBench_dev': {
        'hf_name': 'MagedSaeed/arabench_dev_experimental',
        'tawjeeh_name': 'AraBench_dev',
        'prompt_ids': [14852, 14850, 14789, 14781, 14561],
        'task_type': 'classification',
    },
    'NLI/ArEntail': {
        'hf_name': 'MagedSaeed/ArEntail_experimental',
        'tawjeeh_name': 'ArEntail',
        'prompt_ids': [14581, 14816, 14818, 14819, 14820],
        'task_type': 'classification',
    },
    'NLU/ArabicMMLU': {
        'hf_name': 'MagedSaeed/ArabicMMLU_experimental',
        'tawjeeh_name': 'ArabicMMLU',
        'prompt_ids': [14571, 14869, 14787, 14797, 14798],
        'task_type': 'classification',
    },
    'sarcasm_detection/ArSarcasm_v2': {
        'hf_name': 'MagedSaeed/ArSarcasm_v2_experimental',
        'tawjeeh_name': 'ArSarcasm_v2',
        'prompt_ids': [14779, 14802, 14835, 14837, 14838],
        'task_type': 'classification',
    },
    'machine_translation/opus-100': {
        'hf_name': 'MagedSaeed/opus-100_ar_en_experimental',
        'tawjeeh_name': 'opus-100',
        'prompt_ids': [14684, 14688, 14680, 14682, 14640],
        'task_type': 'generation',
        'preprocess_data': lambda ds: ds.map(
            lambda x: {'translation': {'en': x['en'], 'ar': x['ar']}},
            remove_columns=['en', 'ar'],
        ),
    },
    'summarization/xlsum': {
        'hf_name': 'MagedSaeed/xlsum_arabic_experimental',
        'tawjeeh_name': 'xlsum',
        'prompt_ids': [14871, 14803, 14856, 14858, 14668],
        'task_type': 'generation',
    },
    # ---- Secondary datasets (evaluate only) ----
    'dialect_identification/Arabic_Dialects_Dataset': {
        'hf_name': 'MagedSaeed/arabic_dialects_dataset_experimental',
        'tawjeeh_name': 'Arabic_Dialects_Dataset',
        'prompt_ids': [14102, 14783, 14784, 14790, 14851],
        'task_type': 'classification',
    },
    'NLI/ArabicTE': {
        'hf_name': 'MagedSaeed/ArabicTE_experimental',
        'tawjeeh_name': 'ArabicTE',
        'prompt_ids': [14582, 14673, 14724, 14805, 14855],
        'task_type': 'classification',
    },
    'NLU/belebele': {
        'hf_name': 'MagedSaeed/belebele_experimental',
        'tawjeeh_name': 'belebele',
        'prompt_ids': [14854, 14853, 14801, 14800, 14575],
        'task_type': 'classification',
    },
    'sarcasm_detection/iSarcasmEval_task': {
        'hf_name': 'MagedSaeed/iSarcasmEval_task_A_experimental',
        'tawjeeh_name': 'iSarcasmEval_task_A',
        'prompt_ids': [14602, 14780, 14859, 14860, 14864],
        'task_type': 'classification',
    },
    'machine_translation/tatoeba_mt': {
        'hf_name': 'MagedSaeed/tatoeba_mt_ara_eng_experimental',
        'tawjeeh_name': 'tatoeba_mt',
        'prompt_ids': [14866, 14867, 14868, 14889, 14890],
        'task_type': 'generation',
    },
    'summarization/AraSum': {
        'hf_name': 'MagedSaeed/AraSum_arabic_experimental',
        'tawjeeh_name': 'AraSum',
        'prompt_ids': [14891, 14857, 14736, 14735, 14628],
        'task_type': 'generation',
    },
}

In [9]:
# Template processing functions (matching the tune notebooks)

def preprocess_template_classification(template):
    prefix, suffix = template.split('|||')
    prefix = prefix.replace('\xa0', '')
    return f'{prefix.strip()}\n{suffix.strip()}'  # output on last line

def preprocess_template_generation(template):
    prefix, suffix = template.split('|||')
    prefix = prefix.replace('\xa0', '')
    return f'{prefix.strip()}|||{suffix.strip()}'  # keep ||| separator

def apply_template(prompt_template, sample, task_type):
    template = prompt_template['template']
    if task_type == 'classification':
        template = preprocess_template_classification(template)
    else:
        template = preprocess_template_generation(template)
    sample_copy = dict(sample)
    sample_copy['answer_choices'] = prompt_template.get('answer_choices', [])
    env = Environment(undefined=StrictUndefined)
    rendered = env.from_string(template).render(**sample_copy)
    return rendered

def get_full_text_classification(rendered):
    """For classification tasks, the full text is the rendered template as-is."""
    return rendered

def get_full_text_generation(rendered):
    """For generation tasks, split on ||| and rejoin (prefix + ' ' + suffix)."""
    prefix, suffix = rendered.split('|||')
    prefix = prefix.strip().replace('\xa0', '')
    suffix = suffix.strip()
    return f"{prefix} {suffix}"

In [11]:
results = {}

for dataset_key, cfg in DATASET_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"Dataset: {dataset_key}")
    print(f"{'='*60}")

    # Get prompts for this dataset
    dataset_prompts = [p for p in filtered_prompts if p['id'] in cfg['prompt_ids']]
    print(f"  Prompts found: {len(dataset_prompts)}")

    if len(dataset_prompts) == 0:
        print("  WARNING: No prompts found, skipping")
        continue

    # Load HF dataset
    hf_dataset = datasets.load_dataset(cfg['hf_name'])
    if 'preprocess_data' in cfg:
        hf_dataset = cfg['preprocess_data'](hf_dataset)
    print(f"  Train samples: {len(hf_dataset['train']):,}, Test samples: {len(hf_dataset['test']):,}")

    get_full_text = get_full_text_classification if cfg['task_type'] == 'classification' else get_full_text_generation

    for split_name in ['train', 'test']:
        if split_name not in hf_dataset:
            continue
        split_data = hf_dataset[split_name]
        step_size = max(1, len(split_data) // len(dataset_prompts))

        token_counts = []
        for i, sample in enumerate(tqdm(split_data, desc=f"  {split_name}")):
            prompt_idx = min(i // step_size, len(dataset_prompts) - 1)
            rendered = apply_template(dataset_prompts[prompt_idx], sample, cfg['task_type'])
            full_text = get_full_text(rendered)
            n_tokens = len(tokenizer(full_text, add_special_tokens=True)['input_ids'])
            token_counts.append(n_tokens)

        token_counts = np.array(token_counts)
        exceeding = token_counts > CONTEXT_WINDOW

        key = f"{dataset_key} ({split_name})"
        results[key] = {
            'total': len(token_counts),
            'exceeding': int(exceeding.sum()),
            'pct': exceeding.mean() * 100,
            'min': int(token_counts.min()),
            'max': int(token_counts.max()),
            'mean': float(token_counts.mean()),
            'median': float(np.median(token_counts)),
            'p95': float(np.percentile(token_counts, 95)),
            'p99': float(np.percentile(token_counts, 99)),
        }

        r = results[key]
        print(f"  [{split_name}] Exceeding {CONTEXT_WINDOW}: {r['exceeding']:,}/{r['total']:,} ({r['pct']:.2f}%)")
        print(f"          min={r['min']}, max={r['max']}, mean={r['mean']:.1f}, median={r['median']:.1f}, p95={r['p95']:.0f}, p99={r['p99']:.0f}")


Dataset: dialect_identification/AraBench_dev
  Prompts found: 5
  Train samples: 30,000, Test samples: 10,000


  train:   0%|          | 0/30000 [00:00<?, ?it/s]

  [train] Exceeding 2048: 0/30,000 (0.00%)
          min=50, max=353, mean=108.8, median=102.0, p95=167, p99=210


  test:   0%|          | 0/10000 [00:00<?, ?it/s]

  [test] Exceeding 2048: 0/10,000 (0.00%)
          min=49, max=299, mean=108.4, median=101.0, p95=165, p99=201

Dataset: NLI/ArEntail
  Prompts found: 5
  Train samples: 5,000, Test samples: 1,000


...

  train:   0%|          | 0/30000 [00:00<?, ?it/s]

  [train] Exceeding 2048: 13,250/30,000 (44.17%)
          min=166, max=43737, mean=2480.4, median=1862.0, p95=6548, p99=9447


  test:   0%|          | 0/4689 [00:00<?, ?it/s]

  [test] Exceeding 2048: 2,003/4,689 (42.72%)
          min=369, max=6698, mean=2216.6, median=1869.0, p95=4789, p99=5935

Dataset: dialect_identification/Arabic_Dialects_Dataset
  Prompts found: 5


KeyError: 'train'

In [ ]:
# Summary table
print(f"\nModel: {MODEL_NAME} | Context window: {CONTEXT_WINDOW}")
print(f"\n{'Dataset':<50} {'Total':>8} {'Exceeding':>10} {'%':>8} {'Max Tok':>8}")
print("-" * 88)
for name, r in results.items():
    print(f"{name:<50} {r['total']:>8,} {r['exceeding']:>10,} {r['pct']:>7.2f}% {r['max']:>8,}")

In [ ]:
import matplotlib.pyplot as plt

n_plots = len(results)
ncols = 3
nrows = (n_plots + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5 * nrows))
fig.suptitle(f'Token Count Distributions — {MODEL_NAME} (context = {CONTEXT_WINDOW})', fontsize=14)

# Re-tokenize for histograms (reuse from results loop would be better, but kept simple)
for ax, (name, r) in zip(axes.flat, results.items()):
    # Reconstruct token counts from the stats isn't possible, so we re-tokenize
    # For the histogram we'll use a quick approach: sample from normal approx
    # Actually, let's just plot the stats as a bar + annotation
    stats = [r['min'], r['mean'], r['median'], r['p95'], r['p99'], r['max']]
    labels = ['min', 'mean', 'median', 'p95', 'p99', 'max']
    colors = ['green' if s <= CONTEXT_WINDOW else 'red' for s in stats]
    ax.bar(labels, stats, color=colors, alpha=0.7, edgecolor='black')
    ax.axhline(y=CONTEXT_WINDOW, color='red', linestyle='--', linewidth=2, label=f'Limit ({CONTEXT_WINDOW})')
    ax.set_title(f"{name}\n{r['exceeding']:,}/{r['total']:,} exceeding ({r['pct']:.1f}%)", fontsize=10)
    ax.set_ylabel('Tokens')
    ax.legend(fontsize=8)

# Hide unused subplots
for ax in axes.flat[n_plots:]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()